In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_7.pickle

In [4]:
%%RecordEvent
%%time
### cell 7 ###

# Step 0: filter region to AMERICA (needed by Step 1)
region_filtered = region[
    region.R_NAME == "AMERICA"
][["R_REGIONKEY"]]

# Step 1: get nation keys for America region without an extra merge
america_nations = nation[
    nation.N_REGIONKEY.isin(region_filtered.R_REGIONKEY)
][["N_NATIONKEY"]]

# Step 2: filter customers in America region via a GPU-friendly isin
cust_in_america = customer[
    customer.C_NATIONKEY.isin(america_nations.N_NATIONKEY)
][["C_CUSTKEY"]]

# Step 3: filter orders by date and customer region, add O_YEAR
orders_filtered = (
    orders[
        (orders.O_ORDERDATE >= "1995-01-01")
        & (orders.O_ORDERDATE <  "1997-01-01")
        & orders.O_CUSTKEY.isin(cust_in_america.C_CUSTKEY)
    ]
    .assign(O_YEAR=lambda df: df.O_ORDERDATE.dt.year)
    [["O_ORDERKEY", "O_YEAR"]]
)

# Step 4: bring in supplier nation names in one merge
supplier_nations = (
    supplier[["S_SUPPKEY", "S_NATIONKEY"]]
    .merge(
        nation[["N_NATIONKEY", "N_NAME"]]
              .rename(columns={"N_NAME": "NATION"}),
        left_on="S_NATIONKEY", right_on="N_NATIONKEY"
    )[["S_SUPPKEY", "NATION"]]
)

# Step 5: prefilter parts and compute lineitem volume
part_filtered = part[
    part.P_TYPE == "ECONOMY ANODIZED STEEL"
][["P_PARTKEY"]]

lineitem_filtered = lineitem.assign(
    VOLUME=lineitem.L_EXTENDEDPRICE * (1.0 - lineitem.L_DISCOUNT)
)[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Step 6: chain the remaining joins (3 merges instead of 7)
total = (
    lineitem_filtered
    .merge(orders_filtered,  left_on="L_ORDERKEY", right_on="O_ORDERKEY")
    .merge(part_filtered,    left_on="L_PARTKEY",   right_on="P_PARTKEY")
    .merge(supplier_nations, left_on="L_SUPPKEY",   right_on="S_SUPPKEY")
    [["VOLUME", "O_YEAR", "NATION"]]
)

# Step 7: compute Brazil volume and market share on GPU end-to-end
total = (
    total
    .assign(BRAZIL_VOL=total.VOLUME.where(total.NATION == "BRAZIL", 0))
    .groupby("O_YEAR")[["VOLUME", "BRAZIL_VOL"]]
    .sum()
    .reset_index()
    .assign(MKT_SHARE=lambda df: df.BRAZIL_VOL / df.VOLUME)[["O_YEAR", "MKT_SHARE"]]
    .sort_values("O_YEAR")
)


CPU times: user 130 ms, sys: 40.8 ms, total: 171 ms
Wall time: 193 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high_small/checkpoints/post_cell_7_try_2.pickle